In [4]:
#| default_exp gen

In [1]:
#| hide
import nbdev; nbdev.nbdev_export()

/usr/local/lib/python3.12/dist-packages/nbdev/export.py:80: UserWarning: Notebook '/workspaces/gpt/rugptxl_converter.ipynb' uses `#|export` without `#|default_exp` cell.
Note nbdev2 no longer supports nbdev1 syntax. Run `nbdev_migrate` to upgrade.
See https://nbdev.fast.ai/getting_started.html for more information.
  warn(f"Notebook '{nbname}' uses `#|export` without `#|default_exp` cell.\n"


In [3]:
#| export
def bad_points(tokenizer, point):
    result = []
    for i in range(2, 50):
        code = tokenizer.encode(point*i)
        if len(code) == 1:
            result += [code]
    return result

def bad_words(tokenizer, allow_linebreak):
    bad_symbols = ['[','(','\xa0','*','­', '~', '_', '\\', '\n\n', '\uf04a', '\ufeff', '\u2028']
    bad_words_ids = [tokenizer.encode(s) for s in bad_symbols]
    
    eot = tokenizer.encode('a<|endoftext|>')[1]
    if eot: bad_words_ids += [[eot]]
    
    for point in ['.','*','_','-','\xa0','!']:
        bad_words_ids += bad_points(tokenizer, point)
    linebreaks = [tokenizer.encode(s) for s in ['\n', ' \n']]
    bad_words_ids += [] if allow_linebreak else linebreaks
    bad_words_ids = [sublist for sublist in bad_words_ids if 50257 not in sublist]
    return bad_words_ids

In [6]:
#| export
def iftoken(tokenizer, tokens):
    # returns token id if the given string is one token
    token_ids = [tokenizer.encode(token, add_special_tokens=False) for token in tokens]
    return [id for sublist in token_ids for id in sublist if len(sublist) == 1]

def get_line_enders(tokenizer):
    return [tokenizer.decode(token_id) for token, token_id in tokenizer.get_vocab().items() if '\n' in tokenizer.decode(token_id)]

In [ ]:
#| export
from torch.utils.data import Dataset
import torch

class TextDataset(Dataset):
    def __init__(self, path, tokenizer, seq_length=2048):
        with open(path) as f:
            data = f.read()
        tokens = tokenizer.encode(data)
        examples = []
        for i in range(0, len(tokens) - seq_length + 1, seq_length):
            examples.append(tokens[i:i + seq_length])
        self.samples = torch.LongTensor(examples)
        print('Loaded samples:', len(self.samples))
    
    def __len__(self):
        return len(self.samples)

    def __getitem__(self, item):
        return self.samples[item]